<a href="https://colab.research.google.com/github/vikassinngh123/AI-ML-Learning/blob/main/Machine-Learning/Weather_Logistic_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.linear_model import LogisticRegression #<-Model
from sklearn.model_selection import train_test_split#<-Data spliting
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler

In [2]:
import kagglehub
path = kagglehub.dataset_download("jsphyg/weather-dataset-rattle-package")

Using Colab cache for faster access to the 'weather-dataset-rattle-package' dataset.


#### Data checking and cleaning

In [3]:
raw_df=pd.read_csv(path+'/weatherAUS.csv')
raw_df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [4]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 145460 entries, 0 to 145459
Data columns (total 23 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Date           145460 non-null  object 
 1   Location       145460 non-null  object 
 2   MinTemp        143975 non-null  float64
 3   MaxTemp        144199 non-null  float64
 4   Rainfall       142199 non-null  float64
 5   Evaporation    82670 non-null   float64
 6   Sunshine       75625 non-null   float64
 7   WindGustDir    135134 non-null  object 
 8   WindGustSpeed  135197 non-null  float64
 9   WindDir9am     134894 non-null  object 
 10  WindDir3pm     141232 non-null  object 
 11  WindSpeed9am   143693 non-null  float64
 12  WindSpeed3pm   142398 non-null  float64
 13  Humidity9am    142806 non-null  float64
 14  Humidity3pm    140953 non-null  float64
 15  Pressure9am    130395 non-null  float64
 16  Pressure3pm    130432 non-null  float64
 17  Cloud9am       89572 non-null

In [5]:
raw_df.describe()

,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm
count,143975.000000,144199.000000,142199.000000,82670.000000,75625.000000,135197.000000,143693.000000,142398.000000,142806.000000,140953.000000,130395.00000,130432.000000,89572.000000,86102.000000,143693.000000,141851.00000
mean,12.194034,23.221348,2.360918,5.468232,7.611178,40.035230,14.043426,18.662657,68.880831,51.539116,1017.64994,1015.255889,4.447461,4.509930,16.990631,21.68339
std,6.398495,7.119049,8.478060,4.193704,3.785483,13.607062,8.915375,8.809800,19.029164,20.795902,7.10653,7.037414,2.887159,2.720357,6.488753,6.93665
min,-8.500000,-4.800000,0.000000,0.000000,0.000000,6.000000,0.000000,0.000000,0.000000,0.000000,980.50000,977.100000,0.000000,0.000000,-7.200000,-5.40000
25%,7.600000,17.900000,0.000000,2.600000,4.800000,31.000000,7.000000,13.000000,57.000000,37.000000,1012.90000,1010.400000,1.000000,2.000000,12.300000,16.60000
50%,12.000000,22.600000,0.000000,4.800000,8.400000,39.000000,13.000000,19.000000,70.000000,52.000000,1017.60000,1015.200000,5.000000,5.000000,16.700000,21.10000
75%,16.900000,28.200000,0.800000,7.400000,10.600000,48.000000,19.000000,24.000000,83.000000,66.000000,1022.40000,1020.000000,7.000000,7.000000,21.600000,26.40000
max,33.900000,48.100000,371.000000,145.000000,14.500000,135.000000,130.000000,87.000000,100.000000,100.000000,1041.00000,1039.600000,9.000000,9.000000,40.200000,46.70000


In [6]:
# Removing those colns which have na in raintoday or tommorow

raw_df.dropna(subset=['RainToday','RainTomorrow'],inplace=True)

year=pd.to_datetime(raw_df['Date']).dt.year

raw_df.drop(columns=['Date'],inplace=True)

In [7]:
from pandas.core.arrays import categorical
numeric_cols=raw_df.select_dtypes(include=['float64']).columns.tolist()
categorical_cols=raw_df.select_dtypes(include=['object']).columns.tolist()[0:-1]
print(numeric_cols)
print(categorical_cols)

['MinTemp', 'MaxTemp', 'Rainfall', 'Evaporation', 'Sunshine', 'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm', 'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm', 'Cloud9am', 'Cloud3pm', 'Temp9am', 'Temp3pm']
['Location', 'WindGustDir', 'WindDir9am', 'WindDir3pm', 'RainToday']


In [8]:
## Spliting data for training and test

input_col=(numeric_cols+categorical_cols)

input=raw_df[input_col]
target=raw_df['RainTomorrow']

input_train=input[year<2015]
input_test=input[year>=2015]

target_train=target[year<2015]
target_test=target[year>=2015]

In [9]:
input_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 97988 entries, 0 to 144552
Data columns (total 21 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   MinTemp        97674 non-null  float64
 1   MaxTemp        97801 non-null  float64
 2   Rainfall       97988 non-null  float64
 3   Evaporation    61657 non-null  float64
 4   Sunshine       57942 non-null  float64
 5   WindGustSpeed  91160 non-null  float64
 6   WindSpeed9am   97114 non-null  float64
 7   WindSpeed3pm   96919 non-null  float64
 8   Humidity9am    96936 non-null  float64
 9   Humidity3pm    96872 non-null  float64
 10  Pressure9am    88876 non-null  float64
 11  Pressure3pm    88857 non-null  float64
 12  Cloud9am       63000 non-null  float64
 13  Cloud3pm       61966 non-null  float64
 14  Temp9am        97414 non-null  float64
 15  Temp3pm        97392 non-null  float64
 16  Location       97988 non-null  object 
 17  WindGustDir    91120 non-null  object 
 18  WindDir9am

#### Missing Value

In [10]:
trf1=ColumnTransformer(transformers=[('imputer',SimpleImputer(strategy='median'),[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]),
                                     ('imputer_s',SimpleImputer(strategy='most_frequent'),[16,17,18,19,20])],remainder='passthrough')

#### OneHotEncoder Transformer

In [11]:
trf2=ColumnTransformer(transformers=[('OneHotEncoder',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[16,17,18,19,20])],
                       remainder='passthrough')

#### Scaling

In [12]:
trf3=ColumnTransformer(transformers=[('Scaler',StandardScaler(),slice(0,125))],remainder='passthrough')

#### Model

##### As already seen the Logistic Regression Model is best to use as it take very less time very equally good result as a fine tunnied Forest Model

In [13]:
trf4=LogisticRegression(class_weight={'No':0.95,'Yes':1.85})

### Pipeline Setup

In [14]:
pipe=Pipeline([
              ('trf1',trf1),
              ('trf2',trf2),
              ('trf3',trf3),
              ('trf4',trf4)
               ])

In [15]:
pipe.fit(input_train,target_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('imputer',
                                                  SimpleImputer(strategy='median'),
                                                  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9,
                                                   10, 11, 12, 13, 14, 15]),
                                                 ('imputer_s',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [16, 17, 18, 19, 20])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('OneHotEncoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [16, 17, 18, 19, 20])])),
                ('trf3',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('Scaler', StandardScaler(),
                                                  slice(0, 125, None))])),
                ('trf4',
                 LogisticRegression(class_weight={'No': 0.95, 'Yes': 1.85}))])

In [16]:
pipe.named_steps

{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('imputer', SimpleImputer(strategy='median'),
                                  [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,
                                   14, 15]),
                                 ('imputer_s',
                                  SimpleImputer(strategy='most_frequent'),
                                  [16, 17, 18, 19, 20])]),
 'trf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('OneHotEncoder',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [16, 17, 18, 19, 20])]),
 'trf3': ColumnTransformer(remainder='passthrough',
                   transformers=[('Scaler', StandardScaler(),
                                  slice(0, 125, None))]),
 'trf4': LogisticRegression(class_weight={'No': 0.95, 'Yes': 1.85})}

In [17]:
pred=pipe.predict(input_test)

print(f"Model accuracy {accuracy_score(target_test,pred)*100}")

print(f'\nConfusion Matrix\n{confusion_matrix(target_test,pred,normalize='true')}')

print(f'\nClassification Report\n{classification_report(target_test,pred)}')


Model accuracy 83.76831234374636

Confusion Matrix
[[0.90028746 0.09971254]
 [0.38466447 0.61533553]]

Classification Report
              precision    recall  f1-score   support

          No       0.89      0.90      0.90     33396
         Yes       0.63      0.62      0.62      9403

    accuracy                           0.84     42799
   macro avg       0.76      0.76      0.76     42799
weighted avg       0.84      0.84      0.84     42799



#### Saving the model for future use

In [18]:
import joblib

# Save your entire sequential pipeline to a file
joblib.dump(pipe, 'weather_logistic_pipeline.pkl')
print("Pipeline successfully saved as 'weather_logistic_pipeline.pkl'!")

Pipeline successfully saved as 'weather_logistic_pipeline.pkl'!
